# **For ICANS Group**

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

## **Source EEG based dataframe creation**
**Column names**

**BDSPPatientID, StudyID, Start Time Source File, SourceFilename, ErroneousBDSPInFile, ICANS, ICANSOnsetDate, NoOfDistinctSourceFiles**

In [ ]:
df_icans = pd.read_excel('/home/ayush/Desktop/GitHub-YAMA/ICANS/SLEEPHeadbasedSelectEEGSegments/Spreadsheets/DailyICANSData.xlsx')

df_icans_filenames = pd.read_excel('/home/ayush/Desktop/GitHub-YAMA/ICANS/SLEEPHeadbasedSelectEEGSegments/Spreadsheets/ICANSFiles.xlsx')


# ----------------------Select columns that start with 'Unnamed' and Remove them--------------------------------
unnamed_cols = [col for col in df_icans_filenames.columns if col.startswith('Unnamed')]
df_icans_filenames = df_icans_filenames.loc[:, ~df_icans_filenames.columns.str.startswith('Unnamed')]

#------------------- sort by Study IDs in assending order -------------------------
df_icans_filenames = df_icans_filenames.sort_values(by='StudyID', ascending=True).reset_index(drop=True)

#------------------------ Finding ICANS onset dates for each patient ----------------------------------
print('\n')
print('Finding ICANS onset dates for each patient')
df_icans['DateICANSOnset'] = pd.to_datetime(df_icans['DateICANSOnset'], errors='coerce')
# Group by StudyID, drop NaT, keep only date, join as comma-separated string
dates_per_study = (
    df_icans
    .groupby('StudyID')['DateICANSOnset']
    .apply(lambda x: ', '.join(x.dropna().dt.date.astype(str)))
    .reset_index()
)
dates_per_study.columns = ['StudyID', 'ICANS_Onset_Dates']

# Replace empty strings (no dates) with 'Unknown'
dates_per_study['ICANS_Onset_Dates'] = dates_per_study['ICANS_Onset_Dates'].replace('', 'Unknown')

# Count how many are unknown vs known
summary = dates_per_study['ICANS_Onset_Dates'].apply(lambda x: 'Unknown' if x == 'Unknown' else 'Known').value_counts()
total_patients = len(dates_per_study)
print(f"Total patients: {total_patients}\n")
print(summary)


# ----------------------------- Adding the ICANS Onset Date -----------------------------
# Create a dictionary mapping StudyID → ICANS_Onset_Dates
study_to_dates = dict(zip(dates_per_study['StudyID'], dates_per_study['ICANS_Onset_Dates']))

# Add new column to df_icans_filenames using the mapping
df_icans_filenames['ICANSOnsetDate'] = df_icans_filenames['StudyID'].map(study_to_dates)

def clean_icans_dates(dates_str):
    # If the value is NaN or Unknown, keep as is
    if pd.isna(dates_str) or dates_str == 'Unknown':
        return dates_str
    
    # Split by comma, remove extra spaces
    dates = [d.strip() for d in dates_str.split(',')]    
    # Get unique dates
    unique_dates = sorted(set(dates))   
    # Join back
    return ', '.join(unique_dates)

df_icans_filenames['ICANSOnsetDate'] = df_icans_filenames['ICANSOnsetDate'].apply(clean_icans_dates)

df_icans_filenames = df_icans_filenames.drop(
    columns=["Start Time Segments File", "Segments Filename"],
    errors="ignore"
)

# Finding unique source file per patient
df_icans_filenames["NoOfDistinctSourceFiles"] = (
    df_icans_filenames.groupby("StudyID")["SourceFilename"]
    .transform("nunique")
)

print('\nTotal unique subjects by BDSPID =='+str(df_icans_filenames['BDSPPatientID'].nunique()))
print('Total unique subjects by StidyID =='+str(df_icans_filenames['StudyID'].nunique())+'\n')
df_icans_filenames

## **Adding total No of vanilla valid 10 min segmensts from the SourceEEGFile for every patient**
**Added column names: 'VanillaValidSegmentsFilename', 'NoOfValidVanillaSegments'**

In [ ]:
base_path_to_vanilla_segments = Path(
    "/home/ayush/Desktop/GitHub-YAMA/ICANS/SLEEPHeadbasedSelectEEGSegments/AllContiniousEEGSegments/Morgoth_ready_vanillaEEGSegs/ICANS_group")

def get_segments_for_row(study_id, source_filename):
    # Extract timestamp from source filename
    match = re.search(r'(\d{4}-\d{2}-\d{2})_(\d{6})$', source_filename)
    if not match:
        return []
    
    date_part = match.group(1)
    time_part = match.group(2)
    time_formatted = f"{time_part[:2]}:{time_part[2:4]}:{time_part[4:]}"
    timestamp_for_seg = f"{date_part} {time_formatted}"
    
    # Glob all segments for this study in main folder
    all_seg_files = base_path_to_vanilla_segments.glob(f"{study_id}_seg*_SourceStart_*.mat")
    
    # Filter manually by timestamp and only keep the filename
    matched_files = []
    for f in all_seg_files:
        fname_clean = f.name.replace(":", "").replace(" ", "")
        timestamp_clean = timestamp_for_seg.replace(":", "").replace(" ", "")
        if timestamp_clean in fname_clean:
            matched_files.append(f.name)  # <--- only the filename, not full path
    
    return sorted(matched_files)

# Apply to the entire dataframe
df_icans_filenames['VanillaValidSegmentsFilename'] = df_icans_filenames.apply(
    lambda row: get_segments_for_row(row['StudyID'], row['SourceFilename']),
    axis=1
)

df_icans_filenames['NoOfValidVanillaSegments'] = df_icans_filenames['VanillaValidSegmentsFilename'].apply(len)

# Drop rows where NoOfValidVanillaSegments is 0
df_icans_filenames_filtered = df_icans_filenames[df_icans_filenames['NoOfValidVanillaSegments'] != 0].reset_index(drop=True)


print('\nTotal unique subjects by BDSPID =='+str(df_icans_filenames_filtered['BDSPPatientID'].nunique()))
print('Total unique subjects by StidyID =='+str(df_icans_filenames_filtered['StudyID'].nunique())+'\n')
df_icans_filenames_filtered

## **ICANS Patient-Day Segment Counting**

1. **Parse valid segments:** Extract the quality-passing 10-minute EEG segments listed in `VanillaValidSegmentsFilename` for each source EEG file.

2. **Determine recording dates:** Extract the recording date from `SourceFilename` and group records by `BDSPPatientID` and `RecordingDate` to combine all source files from the same patient-day.

3. **Check ICANS consistency:** Verify that each patient-day has a consistent ICANS grade. Exclude records for which the recording date cannot be determined.

4. **Count all segments:** Combine the valid segments across source files for each patient-day, remove duplicate segment filenames, and calculate the total number of unique valid segments.

5. **Summarize availability:** Report the number of source files, unique patients, patient-days, and valid segments overall, along with segment counts and summary statistics stratified by ICANS grade.

In [ ]:
import ast
import pandas as pd
from pathlib import Path

df_icans_all = df_icans_filenames_filtered.copy()

def parse_segments(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    return []

df_icans_all['VanillaValidSegmentsFilename'] = (
    df_icans_all['VanillaValidSegmentsFilename'].apply(parse_segments)
)

df_icans_all['RecordingDate'] = pd.to_datetime(
    df_icans_all['SourceFilename'].astype(str).str.extract(r'(\d{4}-\d{2}-\d{2})')[0],
    errors='coerce'
).dt.date

if df_icans_all['RecordingDate'].isna().any():
    print("Rows with missing recording dates:")
    print(df_icans_all.loc[
        df_icans_all['RecordingDate'].isna(),
        ['BDSPPatientID', 'SourceFilename']
    ].to_string(index=False))

df_icans_all = df_icans_all[df_icans_all['RecordingDate'].notna()].copy()

grade_check = df_icans_all.groupby(
    ['BDSPPatientID', 'RecordingDate']
)['ICANS'].nunique()

assert (grade_check <= 1).all(), "Some patient-days have conflicting ICANS grades."

df_icans_patientday_all = (
    df_icans_all.groupby(['BDSPPatientID', 'RecordingDate'], as_index=False)
    .agg(
        ICANS=('ICANS', 'first'),
        NoOfSourceFiles=('SourceFilename', 'nunique'),
        SourceFilenames=('SourceFilename', list),
        AllSegments=(
            'VanillaValidSegmentsFilename',
            lambda x: [seg for segment_list in x for seg in segment_list]
        )
    )
)

df_icans_patientday_all['AllSegments'] = (
    df_icans_patientday_all['AllSegments'].apply(
        lambda x: list(dict.fromkeys(x))
    )
)
df_icans_patientday_all['NoOfAllSegments'] = (
    df_icans_patientday_all['AllSegments'].str.len()
)

print("Source-file records:", len(df_icans_all))
print("Unique patients:", df_icans_all['BDSPPatientID'].nunique())
print("Unique patient-days:", len(df_icans_patientday_all))
print("Total source files across patient-days:",
      df_icans_patientday_all['NoOfSourceFiles'].sum())
print("Total unique valid segments across patient-days:",
      df_icans_patientday_all['NoOfAllSegments'].sum())

print("\nValid segments per patient-day:")
print(df_icans_patientday_all['NoOfAllSegments'].describe().to_string())

# print("\nDistribution of valid segments per patient-day:")
# print(
#     df_icans_patientday_all['NoOfAllSegments']
#     .value_counts()
#     .sort_index()
#     .to_string()
# )

print("\nPatient-days by ICANS grade:")
print(
    df_icans_patientday_all.groupby('ICANS').agg(
        PatientDays=('BDSPPatientID', 'size'),
        Patients=('BDSPPatientID', 'nunique'),
        SourceFiles=('NoOfSourceFiles', 'sum'),
        AllValidSegments=('NoOfAllSegments', 'sum'),
        MeanSegmentsPerDay=('NoOfAllSegments', 'mean'),
        MedianSegmentsPerDay=('NoOfAllSegments', 'median'),
        MaxSegmentsPerDay=('NoOfAllSegments', 'max')
    ).to_string()
)

print("\nPatient-days with the most valid segments:")
df_icans_patientday_all


# **For Control Group**

## **Source EEG based dataframe creation**
**Column names**

**BDSPPatientID, StudyID, Start Time Source File, SourceFilename, ErroneousBDSPInFile, ICANS, ICANSOnsetDate, NoOfDistinctSourceFiles**

In [ ]:

from pathlib import Path
import pandas as pd

folder_path = Path("/home/ayush/Desktop/ICANS_dataset/AllContinuousEEGs/SourceFullEEGs/NormFullEEGs")

filenames = sorted(f.stem for f in folder_path.iterdir() if f.is_file())

df_controled = pd.DataFrame({
    'SourceFilename': filenames,
    'BDSPPatientID': [name.split('_')[0] for name in filenames]
})

df_controled['RecordingDate'] = pd.to_datetime(
    df_controled['SourceFilename'].str.extract(r'(\d{4}-\d{2}-\d{2})')[0],
    errors='coerce'
).dt.date

df_controled['NoOfDistinctSourceFiles'] = (
    df_controled.groupby('BDSPPatientID')['SourceFilename'].transform('nunique')
)

df_controled['ICANS'] = 0

df_controled = df_controled[['BDSPPatientID', 'RecordingDate','SourceFilename', 
                               'NoOfDistinctSourceFiles', 'ICANS']]

print("Total files:", len(df_controled))
print("Unique BDSPPatientIDs:", df_controled['BDSPPatientID'].nunique())
print("Missing recording dates:", df_controled['RecordingDate'].isna().sum())

df_controled

## **Adding total No of vanilla valid 10 min segmensts from the SourceEEGFile for every patient**


In [ ]:
base_path_to_vanilla_segments = Path(
    "/home/ayush/Desktop/ICANS_dataset/SLEEPHeadbasedSelectEEGSegments/AllContiniousEEGSegments/Morgoth_ready_vanillaEEGSegs/Normal_group")

def get_segments_for_row(study_id, source_filename):
    # Extract timestamp from source filename
    match = re.search(r'(\d{4}-\d{2}-\d{2})_(\d{6})$', source_filename)
    if not match:
        return []
    
    date_part = match.group(1)
    time_part = match.group(2)
    time_formatted = f"{time_part[:2]}:{time_part[2:4]}:{time_part[4:]}"
    timestamp_for_seg = f"{date_part} {time_formatted}"
    
    # Glob all segments for this study in main folder
    all_seg_files = base_path_to_vanilla_segments.glob(f"{study_id}_seg*_SourceStart_*.mat")
    
    # Filter manually by timestamp and only keep the filename
    matched_files = []
    for f in all_seg_files:
        fname_clean = f.name.replace(":", "").replace(" ", "")
        timestamp_clean = timestamp_for_seg.replace(":", "").replace(" ", "")
        if timestamp_clean in fname_clean:
            matched_files.append(f.name)  # <--- only the filename, not full path
    
    return sorted(matched_files)

# Apply to the entire dataframe
df_controled['VanillaValidSegmentsFilename'] = df_controled.apply(
    lambda row: get_segments_for_row(row['BDSPPatientID'], row['SourceFilename']),
    axis=1
)

df_controled['NoOfValidVanillaSegments'] = df_controled['VanillaValidSegmentsFilename'].apply(len)

# Drop rows where NoOfValidVanillaSegments is 0
df_controled_filtered = df_controled[df_controled['NoOfValidVanillaSegments'] != 0].reset_index(drop=True)


print('Total unique subjects by StidyID =='+str(df_controled_filtered['BDSPPatientID'].nunique())+'\n')
df_controled_filtered

# **ICANS Control Patient-Day Segment Counting**

In [ ]:
import pandas as pd
import ast

df_icans0_all = df_controled_filtered.copy()

def parse_segments(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    return []

df_icans0_all['VanillaValidSegmentsFilename'] = (
    df_icans0_all['VanillaValidSegmentsFilename'].apply(parse_segments)
)

df_icans0_all['RecordingDate'] = pd.to_datetime(
    df_icans0_all['RecordingDate'], errors='coerce'
).dt.date

df_icans0_all = df_icans0_all[
    df_icans0_all['RecordingDate'].notna()
].copy()

df_icans0_all['ICANS'] = 0

df_icans0_patientday_all = (
    df_icans0_all.groupby(
        ['BDSPPatientID', 'RecordingDate'], as_index=False
    )
    .agg(
        ICANS=('ICANS', 'first'),
        NoOfSourceFiles=('SourceFilename', 'nunique'),
        SourceFilenames=('SourceFilename', list),
        AllSegments=(
            'VanillaValidSegmentsFilename',
            lambda x: [seg for segment_list in x for seg in segment_list]
        )
    )
)

df_icans0_patientday_all['AllSegments'] = (
    df_icans0_patientday_all['AllSegments'].apply(
        lambda x: list(dict.fromkeys(x))
    )
)

df_icans0_patientday_all['NoOfAllSegments'] = (
    df_icans0_patientday_all['AllSegments'].str.len()
)

print("Source-file records:", len(df_icans0_all))
print("Unique patients:", df_icans0_all['BDSPPatientID'].nunique())
print("Unique patient-days:", len(df_icans0_patientday_all))
print("Total source files:", df_icans0_patientday_all['NoOfSourceFiles'].sum())
print("Total unique valid segments:", df_icans0_patientday_all['NoOfAllSegments'].sum())

print("\nValid segments per patient-day:")
print(df_icans0_patientday_all['NoOfAllSegments'].describe().to_string())

print("\nPatient-day summary:")
print(
    df_icans0_patientday_all.groupby('ICANS').agg(
        PatientDays=('BDSPPatientID', 'size'),
        Patients=('BDSPPatientID', 'nunique'),
        SourceFiles=('NoOfSourceFiles', 'sum'),
        AllValidSegments=('NoOfAllSegments', 'sum'),
        MeanSegmentsPerDay=('NoOfAllSegments', 'mean'),
        MedianSegmentsPerDay=('NoOfAllSegments', 'median'),
        MaxSegmentsPerDay=('NoOfAllSegments', 'max')
    ).to_string()
)

print("\nPatient-days with the most valid segments:")

df_icans0_patientday_all

# **Transfer Files**

In [ ]:
from tqdm import tqdm
import shutil
from pathlib import Path

# Paths
source_dir = Path("/home/ayush/Desktop/ICANS_dataset/SLEEPHeadbasedSelectEEGSegments/AllContiniousEEGSegments/Morgoth_ready_vanillaEEGSegs/ICANS_group")
dest_dir = Path("/home/ayush/Desktop/NESI_review_material/ICANS_Awake_Segment_Ablation/AllSegmentsAnalysis/AllDayCluster10minSegement")
dest_dir.mkdir(parents=True, exist_ok=True)  # create destination folder if it doesn't exist

# Loop over rows with progress bar
for i, row in tqdm(df_icans_patientday_all.iterrows(), total=len(df_icans_patientday_all), desc="Copying best segments"):
    best_segments = row['AllSegments']
    
    if isinstance(best_segments, list) and len(best_segments) > 0:
        for seg_file in best_segments:
            src_file = source_dir / seg_file
            dst_file = dest_dir / seg_file
            
            if src_file.exists():
                shutil.copy(src_file, dst_file)
            else:
                print(f"Source file not found: {src_file}")

In [ ]:
df_icans_patientday_all['NoOfAllSegments'].sum() + df_icans_patientday_all['NoOfAllSegments'].sum()

# **Make Metadata For Analysis**

In [ ]:
from pathlib import Path
import pandas as pd
import mat73
import numpy as np

def group_icans(y):
    if y in [0]:
        return 0 # Good Neurological state
    elif y in [1, 2]:
        return 1 # Moderate Neurological state
    elif y in [3, 4]:
        return 2 # Bad Neurological state
    else:
        return None 

folder_path = Path("/home/ayush/Desktop/NESI_review_material/ICANS_Awake_Segment_Ablation/AllSegmentsAnalysis/AllDayCluster10minSegement")

records = []
for f in sorted(folder_path.glob("*.mat")):
    mat = mat73.loadmat(f, only_include=['ICANS_value'])
    records.append({
        'MorgothOutputFilename': f.with_suffix('.csv').name,
        'BDSPPatientID': f.stem.split('_')[0],
        'RawScore': int(np.asarray(mat['ICANS_value']).squeeze()),
        'Dataset': 'ICANS'
    })

df_icans_morgoth = pd.DataFrame(records)
df_icans_morgoth = df_icans_morgoth[['BDSPPatientID', 'Dataset', 	
                                     'MorgothOutputFilename', 'RawScore']]

df_icans_morgoth["GroupedScore"] = df_icans_morgoth["RawScore"].apply(group_icans)

print("Total files:", len(df_icans_morgoth))
print("Unique patients:", df_icans_morgoth['BDSPPatientID'].nunique())


df_icans_morgoth

In [ ]:
import pandas as pd

csv_path = "/home/ayush/Desktop/NESI_review_material/Morgoth_vs_NESI_datasplit_Realted/Final_NESIscores_full_dataset_result.csv"

df_split = pd.read_csv(csv_path, low_memory=False)
df_split = df_split[df_split['Dataset'] == 'ICANS'].copy()

df_split['BDSPPatientID'] = df_split['BDSPPatientID'].astype(str).str.strip()
df_icans_morgoth['BDSPPatientID'] = df_icans_morgoth['BDSPPatientID'].astype(str).str.strip()

split_counts = df_split.groupby('BDSPPatientID')['Split'].nunique()
assert (split_counts <= 1).all(), "Some patients belong to multiple splits!"

split_map = df_split.drop_duplicates('BDSPPatientID').set_index('BDSPPatientID')['Split']
df_icans_morgoth['Split'] = df_icans_morgoth['BDSPPatientID'].map(split_map)

print("Total rows:", len(df_icans_morgoth))
print("Unique patients:", df_icans_morgoth['BDSPPatientID'].nunique())
print("\nSplit distribution:")
print(df_icans_morgoth.groupby('Split', dropna=False).agg(
    Rows=('BDSPPatientID', 'size'),
    Patients=('BDSPPatientID', 'nunique')
))


from itertools import combinations

split_ids = {
    s: set(df_icans_morgoth.loc[df_icans_morgoth['Split'] == s, 'BDSPPatientID'])
    for s in ['Train', 'Val', 'Test']
}

for a, b in combinations(split_ids, 2):
    overlap = split_ids[a] & split_ids[b]
    print(f"{a} vs {b}: {len(overlap)} overlapping patients")
    if overlap:
        print("Patient IDs:", sorted(overlap))

In [ ]:
df_icans_morgoth.to_csv('/home/ayush/Desktop/NESI_review_material/ICANS_Awake_Segment_Ablation/AllSegmentsAnalysis/AllSegmenst_DayCluster_ICANS_Metadata.csv',
                       index=False)